# Ch. 1 · Première expérience d’IA scientifique

[![Ouvrir dans Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/benlahmar/IASciences/blob/main/notebooks/ch01-premiere-experience-ia.ipynb)

**IA pour les sciences — M1.** Données Iris et arbre de décision : apprendre à distinguer prédiction et explication.

**Durée indicative :** 1 h 30 (atelier facultatif en autonomie)  
**Statut :** atelier facultatif hors volume horaire (préparation du chapitre 2).

**Objectifs**

- Charger un jeu de données et identifier observations, variables et cible.
- Entraîner un premier classifieur et le comparer à une baseline.
- Constater qu’un score de test dépend du découpage choisi.
- Lire un arbre de décision sans le confondre avec une explication causale.

**Prérequis :** aucun, à part savoir exécuter une cellule Python.

> **Mode d’emploi.** Exécutez les cellules dans l’ordre. Les cellules marquées **✏️ Exercice** contiennent des `...` à compléter : une cellule non complétée lève une erreur, c’est normal. Les corrigés sont repliés sous **« Voir le corrigé »** : cherchez d’abord ! Les cellules suivantes ne dépendent jamais du résultat d’un exercice.

## 1. Les données : qu’est-ce qu’une ligne, qu’est-ce qu’une colonne ?

Le jeu **Iris** (Fisher, 1936) contient 150 fleurs. Chaque **ligne** est une fleur (l’*unité d’observation*),
chaque **colonne** une mesure en centimètres (les *variables explicatives* $X$), et la **cible** $y$ est l’espèce.
Avant tout modèle, on vérifie la taille, les unités et l’équilibre des classes.

In [ ]:
from sklearn.datasets import load_iris
import pandas as pd

iris = load_iris(as_frame=True)
X = iris.data
y = iris.target
print('Dimensions de X :', X.shape)
print('Espèces :', list(iris.target_names))
print('Effectif par classe :', y.value_counts().sort_index().to_dict())
display(X.head())
display(X.describe().round(2))

**🔎 À observer.** Les trois classes ont le même effectif (50) : on dit que le jeu est *équilibré*.
Une exactitude de 33 % correspond donc au **hasard** — c’est le niveau à battre.

## 2. Un premier modèle… et sa référence

Un modèle n’a de valeur que **comparé** à quelque chose. La *baseline* `DummyClassifier` prédit toujours la classe
la plus fréquente : elle ne regarde pas $X$. Nous séparons d’abord les données en **entraînement** (75 %) et **test** (25 %),
avec `stratify=y` pour conserver la proportion des espèces.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, confusion_matrix

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)
baseline = DummyClassifier(strategy='most_frequent').fit(X_train, y_train)
model = DecisionTreeClassifier(max_depth=3, random_state=42).fit(X_train, y_train)

print('Exactitude baseline :', round(accuracy_score(y_test, baseline.predict(X_test)), 3))
pred = model.predict(X_test)
print('Exactitude arbre    :', round(accuracy_score(y_test, pred), 3))
print('Matrice de confusion (lignes = vraie espèce, colonnes = prédiction) :')
print(confusion_matrix(y_test, pred))

## 3. Lire le modèle : un arbre est une suite de questions

Un arbre de décision pose des questions du type « longueur du pétale ≤ 2,45 cm ? ». La figure montre les seuils appris.
Attention : ces seuils **décrivent le modèle**, pas la biologie. Ils disent quelles mesures *séparent* les classes
dans *ce* jeu de données, pas pourquoi les espèces diffèrent.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.tree import plot_tree

fig, ax = plt.subplots(figsize=(11, 6))
plot_tree(model, feature_names=X.columns, class_names=iris.target_names,
          filled=True, rounded=True, fontsize=9, ax=ax)
ax.set_title('Arbre de décision (profondeur 3) appris sur l’entraînement')
plt.show()

## 4. Démonstration : un seul test ne suffit pas

Le score précédent dépend du **tirage** du découpage. Répétons l’expérience avec 50 découpages différents :
la dispersion des scores montre l’incertitude d’une évaluation sur seulement 38 fleurs de test.

In [ ]:
import numpy as np

scores = []
for graine in range(50):
    Xa, Xb, ya, yb = train_test_split(X, y, test_size=0.25, random_state=graine, stratify=y)
    m = DecisionTreeClassifier(max_depth=3, random_state=0).fit(Xa, ya)
    scores.append(m.score(Xb, yb))
scores = np.array(scores)
print(f'Exactitude : moyenne {scores.mean():.3f}, min {scores.min():.3f}, max {scores.max():.3f}')
plt.hist(scores, bins=10, edgecolor='white')
plt.xlabel('Exactitude sur le test'); plt.ylabel('Nombre de découpages')
plt.title('Même modèle, 50 découpages : le score varie')
plt.show()

**🔎 À retenir.** Annoncer le score d’un unique découpage avec trois chiffres significatifs serait trompeur : il varie de plusieurs points d’un tirage à l’autre. On préférera la
**validation croisée** (chapitre 3) et un intervalle (chapitre 7).

### ✏️ Exercice 1 — Calculer l’exactitude à la main

À partir de la matrice de confusion `cm`, calculez l’exactitude sans utiliser `accuracy_score` : nombre de prédictions correctes (diagonale) divisé par le nombre total.

In [ ]:
cm = confusion_matrix(y_test, pred)
corrects = ...          # somme de la diagonale (indice : np.trace)
total = ...             # nombre total d'observations de test
print('Exactitude calculée :', corrects / total)

<details><summary><b>Voir le corrigé</b></summary>

```python
cm = confusion_matrix(y_test, pred)
corrects = np.trace(cm)
total = cm.sum()
print('Exactitude calculée :', corrects / total)
```

La diagonale contient les fleurs bien classées ; le reste de la matrice indique *quelles* espèces sont confondues (souvent *versicolor* et *virginica*, morphologiquement proches).

</details>

### ✏️ Exercice 2 — Profondeur de l’arbre et surapprentissage

Pour chaque profondeur, calculez l’exactitude sur l’**entraînement** et sur le **test**. À partir de quelle profondeur l’arbre « apprend-il par cœur » ? Pourquoi ne faut-il pas choisir la profondeur en regardant le test ?

In [ ]:
for profondeur in [1, 2, 3, 5, None]:
    m = DecisionTreeClassifier(max_depth=profondeur, random_state=42)
    m.fit(X_train, y_train)
    score_train = ...
    score_test = ...
    print(profondeur, round(score_train, 3), round(score_test, 3))

<details><summary><b>Voir le corrigé</b></summary>

```python
for profondeur in [1, 2, 3, 5, None]:
    m = DecisionTreeClassifier(max_depth=profondeur, random_state=42)
    m.fit(X_train, y_train)
    score_train = m.score(X_train, y_train)
    score_test = m.score(X_test, y_test)
    print(profondeur, round(score_train, 3), round(score_test, 3))
```

Sans limite de profondeur, l’exactitude d’entraînement atteint 1,0 : l’arbre isole chaque fleur. Le test ne progresse plus, voire baisse. Choisir la profondeur sur le test reviendrait à l’utiliser pour l’apprentissage : le score annoncé serait alors optimiste. On utilise pour cela une validation croisée sur l’entraînement.

</details>

## Interprétation scientifique (à rédiger)

1. Quelle est la performance de l’arbre comparée à la baseline ?
2. Quelles espèces sont confondues, et est-ce plausible biologiquement ?
3. Pourquoi ce résultat ne se généralise-t-il pas « aux fleurs du monde » ?
4. Les seuils de l’arbre constituent-ils une explication de la différence entre espèces ?